In [ ]:
#Load packages
import pandas as pd
import numpy as np

In [ ]:
df=pd.read_csv('emotion_dataset_raw.csv')
# df.columns = ['emotion', 'text']
df.head()

In [ ]:
df.shape

In [ ]:
#value counts of emotions
df['Emotion'].value_counts()

In [ ]:
#using seaborn to plot
import seaborn as sns
import matplotlib.pyplot as plt
plt.figure(figsize=(10,5))
sns.countplot(x='Emotion', data=df)
plt.show()

In [ ]:
import nltk
nltk.download('wordnet')
nltk.download('stopwords')

from nltk.stem import WordNetLemmatizer #Import the WordNetLemmatizer
import re
from nltk.corpus import stopwords
lemmatizer = WordNetLemmatizer()

# def lemmatization(text):
#     text = text.split()
#     text = [lemmatizer.lemmatize(word) for word in text]
#     return " ".join(text)

def remove_stop_words(text):
    stop_words = set(stopwords.words('english'))
    text = [word for word in text.split() if word.lower() not in stop_words]
    return " ".join(text)

def remove_numbers(text):
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    return text.lower()

def remove_punctuations(text):
    text = re.sub(r'[^\w\s]', '', text)
    return text

def remove_urls(text):
    return re.sub(r'http\S+|www\.\S+', '', text)

# def remove_small_sentences(text):
#     sentences = text.split('.')
#     filtered_sentences = [sentence for sentence in sentences if len(sentence.split()) >= 3]
#     return '. '.join(filtered_sentences)

# df['clean_text'] = df['text'].apply(lemmatization)
df['clean_text'] = df['Text'].apply(remove_stop_words)
# df['clean_text'] = df['clean_text'].apply(remove_small_sentences)

In [ ]:
df['clean_text'] = df['clean_text'].apply(remove_numbers)


In [ ]:
df['clean_text'] = df['clean_text'].apply(lower_case)


In [ ]:
df['clean_text'] = df['clean_text'].apply(remove_punctuations)
df['clean_text'] = df['clean_text'].apply(remove_urls)

In [ ]:
df

In [ ]:
# Check for null values in the 'clean_text' column
null_values = df['clean_text'].isnull().sum()
print(f"Number of null values in 'clean_text' column: {null_values}")

# Optionally, you can also check for empty strings
empty_strings = (df['clean_text'] == '').sum()
print(f"Number of empty strings in 'clean_text' column: {empty_strings}")


In [ ]:
# Drop rows where 'clean_text' is null or empty
df = df.dropna(subset=['clean_text'])
df = df[df['clean_text'] != '']
df = df[df['clean_text'].str.strip() != ""] #remove empty strings after removing spaces


In [ ]:
empty_strings = (df['clean_text'] == '').sum()
print(f"Number of empty strings in 'clean_text' column: {empty_strings}")

In [ ]:
df.shape

In [ ]:
df

In [ ]:
# #Labels encode the target variable
# from sklearn import preprocessing # import the preprocessing module from sklearn
# object = preprocessing.LabelEncoder()
# df['Emotion'] = object.fit_transform(df['Emotion'])

In [ ]:
df

In [ ]:
# data after encoding.
df['Emotion'].value_counts()

In [ ]:
x = df['clean_text']
y = df['Emotion']

In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.3,random_state=42)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

In [ ]:
pipe_lr = Pipeline(steps=[('cv',CountVectorizer()),('lr',LogisticRegression())])
pipe_lr.fit(x_train,y_train)
pipe_lr.score(x_test,y_test)

In [ ]:
pipe_svm = Pipeline(steps=[('cv',CountVectorizer()),('svc', SVC(kernel = 'rbf', C = 10))])
pipe_svm.fit(x_train,y_train)
pipe_svm.score(x_test,y_test)

In [ ]:
pipe_rf = Pipeline(steps=[('cv',CountVectorizer()),('rf', RandomForestClassifier(n_estimators=10))])
pipe_rf.fit(x_train,y_train)
pipe_rf.score(x_test,y_test)

In [ ]:
exp = "i am excited after completing the project"
pipe_lr.predict([exp])

In [ ]:
import joblib
pipeline_file = open("Emotion_pred.pkl","wb")
joblib.dump(pipe_lr,pipeline_file)
pipeline_file.close()